# 가이던스

- 가이던스(guidance): 생성할 때 노이즈 예측을 고쳐서, 조건 y 쪽으로 더 강하게 이동하게 만드는 기법. 학습한 모델은 그대로 두고 샘플링 계산만 바꾼다.
- 필요한 이유: 조건부 확산 모델은 레이블을 입력으로 받지만, 한 단계에서 레이블이 노이즈 예측을 바꾸는 정도가 작다. 그래서 레이블과 다른 숫자나 숫자로 알아보기 어려운 모양이 나오기도 한다.
- 스코어 ∇log p(x_t): x_t를 확률이 높은 쪽으로 옮기는 방향
  - 노이즈 예측과의 관계: ε̂ = −√(1−ᾱ_t) · ∇log p(x_t)
  - denoise에서 ε̂를 빼는 계산은 스코어 방향으로 이동하는 계산과 같다.
- 조건부 스코어를 베이즈 정리로 나누면 두 항이 된다.
  - ∇log p(x_t | y) = ∇log p(x_t) + ∇log p(y | x_t)
  - 첫째 항은 조건과 관계없이 자연스러운 이미지 쪽으로 가는 방향이다.
  - 둘째 항은 레이블 y로 분류될 확률이 높아지는 방향이다. p(y)는 x_t와 관계없어서 기울기가 0이므로 식에서 빠진다.
- 가이던스 세기 γ: 둘째 항에 γ를 곱해 조건 쪽 이동을 키운다.
  - ∇log p(x_t) + γ · ∇log p(y | x_t)
  - γ = 1이면 원래의 조건부 분포이고, γ > 1이면 레이블에 더 잘 맞는 대신 결과의 다양성이 줄어든다.
- 둘째 항을 어떻게 구하느냐에 따라 분류기 가이던스와 분류기 없는 가이던스로 나뉜다.

## 분류기 가이던스 (classifier guidance)

- 노이즈가 섞인 x_t와 t를 받아 레이블을 맞히는 분류기 p_φ(y | x_t)를 따로 학습하고, 그 분류기의 기울기로 둘째 항을 계산한다.
- 생성할 때 노이즈 예측을 다음처럼 바꾼다.
  - ε̃ = ε̂(x_t) − γ · √(1−ᾱ_t) · ∇_x log p_φ(y | x_t)
  - 위의 스코어 식에 ε̂ = −√(1−ᾱ_t) · ∇log p를 대입한 결과다.
- 한계
  - 깨끗한 이미지로 학습한 분류기는 쓸 수 없고, 모든 t의 노이즈 이미지로 분류기를 새로 학습해야 한다.
  - 생성 단계마다 분류기를 역전파해 입력에 대한 기울기를 구해야 한다.
  - 분류기의 기울기에는 이미지를 자연스럽게 만드는 방향보다 분류기 점수만 올리는 방향이 섞일 수 있다.

## 정리: 조건부 확산 모델에서 가이던스로

1. 문제: 조건부 확산 모델은 레이블을 입력으로 받지만, 한 단계에서 레이블이 노이즈 예측을 바꾸는 정도가 작다. 그래서 생성 결과가 레이블을 따르지 않을 때가 있다.
2. 목적: 학습 목표는 노이즈 예측 그대로 두고, 생성할 때 계산만 바꿔서 조건을 더 강하게 반영한다.
3. 원리: 노이즈 예측 ε̂는 스코어에 t마다 다른 음수 −√(1−ᾱ_t)를 곱한 값이다. 그래서 스코어의 레이블 쪽 항 ∇log p(y | x_t)를 키우면 노이즈 예측을 조건 쪽으로 고칠 수 있다.
4. 레이블 쪽 항을 구하는 방법
   - 분류기 가이던스: 노이즈 이미지로 분류기를 따로 학습하고, 입력 x_t에 대한 log p_φ(y | x_t)의 기울기를 구한다. 이 기울기에 γ · √(1−ᾱ_t)를 곱해 노이즈 예측에서 뺀다.
   - 분류기 없는 가이던스: 학습할 때 레이블의 10%를 ∅로 바꿔서, 모델 하나가 조건부 예측과 무조건 예측을 모두 하게 한다. 생성할 때는 두 예측의 차이를 레이블 쪽 방향으로 쓴다.
   - 코드의 `p_uncond`는 레이블을 ∅로 바꾸는 확률이며, 가이던스 세기가 아니다.
5. 세기 w: CFG 식을 ε̃ = (1−w) · ε̂(x_t, ∅) + w · ε̂(x_t, y)로 바꿔 쓰면 w는 조건부 예측의 가중치다. 코드에서는 생성할 때만 쓰는 `guidance_scale`이다.

   | w | 생성 결과 |
   |---|---|
   | 0 | 같은 모델의 무조건 예측만 쓰므로, 무조건 확산 모델과 같은 역할을 한다. |
   | 1 | 01의 조건부 확산 모델과 같다. |
   | 1보다 큼 | 무조건 예측의 가중치 1−w가 음수가 되어, 무조건 예측에서 멀어지는 쪽으로 더 이동한다. |

6. 대가: w가 커질수록 레이블과 잘 맞지만 생성 결과가 서로 비슷해지고, w가 너무 크면 이미지가 어색해진다.

## 분류기 없는 가이던스 (classifier-free guidance, CFG)

- 분류기 없이, 모델 하나가 조건부 예측 ε̂(x_t, y)와 무조건 예측 ε̂(x_t, ∅)를 모두 하도록 학습한다.
- 둘째 항을 구하는 방법: 베이즈 정리를 거꾸로 쓰면 ∇log p(y | x_t) = ∇log p(x_t | y) − ∇log p(x_t)
  - 오른쪽의 두 스코어는 같은 모델의 조건부 예측과 무조건 예측으로 얻는다.
  - 분류기 대신 두 예측의 차이가 "레이블 y 쪽 방향"이 된다.
- 학습
  - 배치마다 레이블의 10%를 "조건 없음" 레이블 ∅로 바꾼다.
  - 이 노트북에서는 레이블 10을 ∅로 쓰므로, 임베딩만 `nn.Embedding(11, 100)`으로 한 행 늘린다. UNet 구조와 손실은 01과 같다.
- 생성: 단계마다 노이즈 예측을 다음 식으로 바꾼다.
  - ε̃ = ε̂(x_t, ∅) + w · (ε̂(x_t, y) − ε̂(x_t, ∅))
  - w = 0이면 무조건 생성이고, w = 1이면 01의 조건부 생성과 같다.
  - w > 1이면 조건이 있을 때와 없을 때의 차이를 w배로 키운다. 가이던스 식의 γ가 w에 해당한다.
- 비용: 단계마다 UNet을 조건 있음과 없음으로 두 번 계산한다. 아래 코드는 두 입력을 배치 하나로 묶어 한 번 호출한다.
- 한계: w를 키우면 레이블에는 잘 맞지만 생성 결과가 서로 비슷해진다. w가 너무 크면 획이 지나치게 굵어지는 것처럼 이미지가 어색해질 수 있다.
- Stable Diffusion과의 관계: v1-4도 캡션의 10%를 빈 문자열로 바꿔 학습했고, `guidance_scale=7.5`가 w다.

### 예제 코드
- 01의 `ConvBlock`, `UNetCond`, `add_noise`는 그대로 쓴다.
- 바뀌는 곳
  - 학습 루프: 레이블을 확률 `p_uncond`로 `NULL_LABEL`로 바꾼다.
  - `Diffuser.guided_noise`: 조건부 예측과 무조건 예측을 섞는다.
  - `Diffuser.sample`: 가이던스 세기 `guidance_scale`을 받는다.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm

NUM_CLASSES = 10
NULL_LABEL = NUM_CLASSES  # 레이블 0~9 다음 번호인 10을 '조건 없음' ∅로 쓴다.


def pos_encoding(ts, output_dim, device='cpu'):
    i = torch.arange(output_dim, device=device)
    div_term = 10000 ** (i / output_dim)
    angles = ts.to(device).float()[:, None] / div_term[None, :]
    v = torch.zeros(len(ts), output_dim, device=device)
    v[:, 0::2] = torch.sin(angles[:, 0::2])
    v[:, 1::2] = torch.cos(angles[:, 1::2])
    return v


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_embed_dim):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )
        self.mlp = nn.Sequential(
            nn.Linear(time_embed_dim, in_ch),
            nn.ReLU(),
            nn.Linear(in_ch, in_ch)
        )

    def forward(self, x, v):
        N, C, _, _ = x.shape
        v = self.mlp(v)
        v = v.view(N, C, 1, 1)
        y = self.convs(x + v)
        return y


class UNetCond(nn.Module):
    def __init__(self, in_ch=1, time_embed_dim=100, num_labels=None):
        super().__init__()
        self.time_embed_dim = time_embed_dim

        self.down1 = ConvBlock(in_ch, 64, time_embed_dim)
        self.down2 = ConvBlock(64, 128, time_embed_dim)
        self.bot1 = ConvBlock(128, 256, time_embed_dim)
        self.up2 = ConvBlock(256 + 128, 128, time_embed_dim)
        self.up1 = ConvBlock(128 + 64, 64, time_embed_dim)
        self.out = nn.Conv2d(64, in_ch, 1)

        self.maxpool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear')

        if num_labels is not None:
            self.label_emb = nn.Embedding(num_labels, time_embed_dim)

    def forward(self, x, timesteps, labels=None):
        t = pos_encoding(timesteps, self.time_embed_dim, device=x.device)
        if labels is not None:
            t += self.label_emb(labels)

        x1 = self.down1(x, t)
        x = self.maxpool(x1)
        x2 = self.down2(x, t)
        x = self.maxpool(x2)

        x = self.bot1(x, t)

        x = self.upsample(x)
        x = torch.cat([x, x2], dim=1)
        x = self.up2(x, t)
        x = self.upsample(x)
        x = torch.cat([x, x1], dim=1)
        x = self.up1(x, t)

        x = self.out(x)
        return x


class Diffuser:
    def __init__(self, num_timesteps=1000, beta_start=0.0001, beta_end=0.02, device='cpu'):
        self.num_timesteps = num_timesteps
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, x_0, t):
        T = self.num_timesteps
        assert (t >= 1).all() and (t <= T).all()
        alpha_bar = self.alpha_bars[t - 1].view(-1, 1, 1, 1)
        noise = torch.randn_like(x_0, device=self.device)
        x_t = torch.sqrt(alpha_bar) * x_0 + torch.sqrt(1 - alpha_bar) * noise
        return x_t, noise

    # CFG 노이즈 예측 ε̃ = ε̂(x_t, ∅) + w·(ε̂(x_t, y) − ε̂(x_t, ∅))를 계산하고, guidance_scale이 w다.
    def guided_noise(self, model, x, t, labels, guidance_scale):
        null_labels = torch.full_like(labels, NULL_LABEL)
        # eval 상태의 BatchNorm은 누적 통계를 쓰므로, 조건부 입력과 무조건 입력을 한 배치로 묶어 UNet을 한 번만 호출해도 결과가 같다.
        eps = model(torch.cat([x, x]), torch.cat([t, t]), torch.cat([labels, null_labels]))
        eps_cond, eps_uncond = eps.chunk(2)
        # 두 예측의 차이가 레이블 y 쪽 방향이고, 그 방향을 w배로 키워 무조건 예측에 더한다.
        return eps_uncond + guidance_scale * (eps_cond - eps_uncond)

    def denoise(self, model, x, t, labels, guidance_scale):
        T = self.num_timesteps
        assert (t >= 1).all() and (t <= T).all()

        t_idx = t - 1
        alpha = self.alphas[t_idx].view(-1, 1, 1, 1)
        alpha_bar = self.alpha_bars[t_idx].view(-1, 1, 1, 1)
        alpha_bar_prev = self.alpha_bars[t_idx - 1].view(-1, 1, 1, 1)

        model.eval()
        with torch.no_grad():
            # 01의 model(x, t, labels) 자리에 가이던스를 적용한 노이즈 예측을 쓴다.
            eps = self.guided_noise(model, x, t, labels, guidance_scale)
        model.train()

        noise = torch.randn_like(x, device=self.device)
        noise[t == 1] = 0

        mu = (x - ((1 - alpha) / torch.sqrt(1 - alpha_bar)) * eps) / torch.sqrt(alpha)
        std = torch.sqrt((1 - alpha) * (1 - alpha_bar_prev) / (1 - alpha_bar))
        return mu + noise * std

    # guidance_scale은 생성할 때만 쓰는 값이므로 학습 코드에는 나오지 않는다.
    def sample(self, model, labels, guidance_scale, x_shape=(1, 28, 28)):
        batch_size = len(labels)
        x = torch.randn((batch_size, *x_shape), device=self.device)

        for i in tqdm(range(self.num_timesteps, 0, -1)):
            t = torch.full((batch_size,), i, device=self.device, dtype=torch.long)
            x = self.denoise(model, x, t, labels, guidance_scale)
        return x.clamp(0, 1)

In [ ]:
batch_size = 128
num_timesteps = 1000
epochs = 10
lr = 1e-3
p_uncond = 0.1  # 레이블을 ∅로 바꿀 확률이며, 가이던스 세기가 아니라 무조건 예측 ε̂(x_t, ∅)를 함께 학습시키는 값이다.
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device('cpu')

dataset = torchvision.datasets.MNIST(root='./data', download=True, transform=transforms.ToTensor())
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

diffuser = Diffuser(num_timesteps, device=device)
model = UNetCond(num_labels=NUM_CLASSES + 1).to(device)  # 레이블 0~9와 ∅까지 임베딩 행이 11개 필요하다.
optimizer = Adam(model.parameters(), lr=lr)

losses = []
for epoch in range(epochs):
    loss_sum = 0.0
    cnt = 0

    for images, labels in tqdm(dataloader):
        optimizer.zero_grad()
        x = images.to(device)
        labels = labels.to(device)
        # 배치의 약 10%만 레이블을 ∅로 바꾸고, 나머지는 01과 같은 조건부 학습이다.
        is_dropped = torch.rand(len(labels), device=device) < p_uncond
        labels = torch.where(is_dropped, NULL_LABEL, labels)
        t = torch.randint(1, num_timesteps + 1, (len(x),), device=device)

        x_noisy, noise = diffuser.add_noise(x, t)
        noise_pred = model(x_noisy, t, labels)
        loss = F.mse_loss(noise, noise_pred)

        loss.backward()
        optimizer.step()

        loss_sum += loss.item()
        cnt += 1

    losses.append(loss_sum / cnt)
    print(f'Epoch {epoch} | Loss: {losses[-1]}')

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

### 가이던스 세기 비교
- 같은 시작 노이즈에서 w만 바꿔 레이블 0~9를 두 번씩 생성한다.
- w = 0은 레이블을 무시한 무조건 생성이고, w = 1은 01 노트북의 조건부 생성과 같다.

In [ ]:
# 가이던스 세기 w 목록이며, 0은 무조건 생성, 1은 01의 조건부 생성과 같고, 1보다 크면 레이블 쪽 차이를 w배로 키운다.
guidance_scales = [0.0, 1.0, 3.0, 7.0]
labels = torch.arange(NUM_CLASSES, device=device).repeat(2)

rows = []
for guidance_scale in guidance_scales:
    torch.manual_seed(0)  # w만 다르고 시작 노이즈와 단계별 노이즈는 같도록 맞춘다.
    rows.append(diffuser.sample(model, labels, guidance_scale).cpu())

fig, axes = plt.subplots(len(guidance_scales), len(labels), figsize=(len(labels), len(guidance_scales) * 1.2))
for r, (guidance_scale, images) in enumerate(zip(guidance_scales, rows)):
    for c, image in enumerate(images):
        ax = axes[r, c]
        ax.imshow(image[0], cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])
        if r == 0:
            ax.set_title(labels[c].item())
    axes[r, 0].set_ylabel(f'w={guidance_scale}')
plt.tight_layout()
plt.show()